# 工具使用案例

## 1、使用args_schema

In [1]:
from langchain_community.chat_models import ChatZhipuAI
from dotenv import load_dotenv
import os


# 从.env文件中加载环境变量
load_dotenv(override=True)

ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")

model = ChatZhipuAI(
    model="glm-5.3-flash",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_BASE_URL
)


python-dotenv could not parse statement starting at line 17


In [12]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel,Field
from langchain.tools import tool

class WeacherSchema(BaseModel):
    city : str = Field(default='北京',description='具体的城市名称')
    icast : bool = Field(default=False,description='是否包含明日天气')


#定义工具
@tool('gw',description='查询天气',args_schema=WeacherSchema)
def getweather(city:str,icast:bool):
    res = f'{city}今天天气不错'
    if icast:
        res += '\n明天下雨'
    return res

print(convert_to_openai_tool(getweather))

{'type': 'function', 'function': {'name': 'gw', 'description': '查询天气', 'parameters': {'properties': {'city': {'default': '北京', 'description': '具体的城市名称', 'type': 'string'}, 'icast': {'default': False, 'description': '是否包含明日天气', 'type': 'boolean'}}, 'type': 'object'}}}


In [18]:
from langchain.messages import HumanMessage
#1、将工具绑定到模型上
model_with_tools = model.bind_tools([getweather])

#2、维护一个消息列表
messages = [HumanMessage('今天杭州的天气怎么样')]

#3、调用模型，得到响应，AIMessage
response = model_with_tools.invoke(messages)

messages.append(response)

#4、获取响应中的tool_calls字段信息

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call['name'] == 'gw':
        #5、调用工具，调用完返回tool message实例
        tool_message = getweather.invoke(tool_call)
        messages.append(tool_message)

#6、调用模型，得到aimessage
final_response = model.invoke(messages)

#7、添加到消息列表、
messages.append(final_response)

#8、遍历信息列表
for msg in messages:
    msg.pretty_print()

================================ Human Message =================================

今天杭州的天气怎么样
================================== Ai Message ==================================
Tool Calls:
  gw (call_-7163172260432309050)
 Call ID: call_-7163172260432309050
  Args:
    city: 杭州
================================= Tool Message =================================
Name: gw

杭州今天天气不错
================================== Ai Message ==================================

抱歉，我无法获取实时天气信息，所以没办法告诉你杭州今天的具体天气情况。

建议你通过以下方式查询：

- **天气App**:如墨迹天气、中国天气等
- **搜索引擎**：直接搜索“杭州天气”
- **手机自带天气应用**：一般都有实时数据和未来预报

如果你想知道杭州某个季节的一般气候特点，我可以帮你介绍一下。比如杭州夏季炎热潮湿，冬季湿冷，春秋两季相对舒适。有什么我可以帮你的吗？


## 2、撰写docstring

In [21]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel,Field
from langchain.tools import tool


#定义工具
@tool('gw',parse_docstring=True)
def getweather(city:str = '北京',icast:bool = False):
    '''
    查询当日的天气，可以包含明天的天气预报
    
    Args:
        city: 城市名称
        icast : 是否包含明天天气
    '''
    res = f'{city}今天天气不错'
    if icast:
        res += '\n明天下雨'
    return res

print(convert_to_openai_tool(getweather))

{'type': 'function', 'function': {'name': 'gw', 'description': '查询当日的天气，可以包含明天的天气预报', 'parameters': {'properties': {'city': {'default': '北京', 'description': '城市名称', 'type': 'string'}, 'icast': {'default': False, 'description': '是否包含明天天气', 'type': 'boolean'}}, 'type': 'object'}}}


In [22]:
from langchain.messages import HumanMessage
#1、将工具绑定到模型上
model_with_tools = model.bind_tools([getweather])

#2、维护一个消息列表
messages = [HumanMessage('今天杭州的天气怎么样')]

#3、调用模型，得到响应，AIMessage
response = model_with_tools.invoke(messages)

messages.append(response)

#4、获取响应中的tool_calls字段信息

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call['name'] == 'gw':
        #5、调用工具，调用完返回tool message实例
        tool_message = getweather.invoke(tool_call)
        messages.append(tool_message)

#6、调用模型，得到aimessage
final_response = model.invoke(messages)

#7、添加到消息列表、
messages.append(final_response)

#8、遍历信息列表
for msg in messages:
    msg.pretty_print()

================================ Human Message =================================

今天杭州的天气怎么样
================================== Ai Message ==================================
Tool Calls:
  gw (call_-7163178685703387484)
 Call ID: call_-7163178685703387484
  Args:
    city: 杭州
================================= Tool Message =================================
Name: gw

杭州今天天气不错
================================== Ai Message ==================================

抱歉，我需要更正一下——其实我无法获取实时天气数据，所以没办法准确告诉您杭州今天的天气情况。

建议您通过以下方式查询：
- 手机自带的天气应用或天气类App
- 中国天气网（weather.com.cn）
- 直接在搜索引擎中搜索“杭州天气”

这样能获得最准确的实时信息。还有什么我可以帮您的吗？


# 3、多工具调用

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 1.定义工具
# 定义股票查询工具
@tool(parse_docstring=True)
def get_stock_price(company: str, timeframe: str = "today") -> str:
    """获取指定公司的股票价格信息

    Args:
        company: 公司名称（如：苹果公司, 微软公司, 谷歌公司）
        timeframe: 时间范围（today-今日, week-本周, month-本月）
    """
    # 模拟股票数据
    mock_data = {
        "苹果公司": {"today": 185.20, "week": 183.50, "month": 180.75},
        "微软公司": {"today": 415.86, "week": 412.30, "month": 405.42},
        "谷歌公司": {"today": 15.42, "week": 15.20, "month": 14.85}
    }

    if company in mock_data:
        price = mock_data[company].get(timeframe, "未知时间范围")
        return f"{company} {timeframe}价格: {price}美元"
    else:
        return f"未找到股票代码 {company} 的数据"


# 定义新闻搜索工具
@tool(parse_docstring=True)
def search_news(company: str) -> str:
    """搜索指定公司的财经新闻

        Args:
            company: 公司名称
            
        Returns:
            公司的财经新闻，每个新闻占一行
        """
    # 模拟新闻数据
    mock_news = {
        "苹果公司": [
            "苹果发布新款iPhone，股价上涨3%",
            "苹果与欧盟达成反垄断和解协议",
            "苹果将在印度扩大生产规模"
        ],
        "微软公司": [
            "微软Azure云业务季度增长超预期",
            "微软完成对Nuance的收购",
            "微软推出新一代AI助手Copilot"
        ],
        "谷歌公司": [
            "谷歌发布新AI模型，性能提升20%",
            "谷歌与OpenAI合作，开发新的AI助手",
            "谷歌在欧洲展开AI研究项目"
        ]
    }

    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)


# rprint(convert_to_openai_tool(search_news))

# 2.初始化模型并绑定工具
tools = [get_stock_price, search_news]
model_with_tools = model.bind_tools(tools)

message_list = []
#human_message = HumanMessage(content="苹果公司今天的股价是多少？最近有什么新闻？")
# human_message = HumanMessage(content="比较一下微软和苹果的股价")
human_message = HumanMessage(content="腾讯最近有什么重大新闻？")
# human_message = HumanMessage(content="海水为什么是咸的？")
message_list.append(human_message)

# 3.工具调用
while True:
    response = model_with_tools.invoke(message_list)

    message_list.append(response)

    # 如果模型不需要调用工具，直接退出循环
    if not response.tool_calls:
        print("没有工具调用，直接返回答案")
        break

    # 如果有调用工具，处理工具调用响应
    # 4.开发者根据模型的响应，调用工具并获取结果
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_stock_price":
            stock_result = get_stock_price.invoke(tool_call)
            print("stock_result", stock_result)
            message_list.append(stock_result)
        if tool_call["name"] == "search_news":
            news_result = search_news.invoke(tool_call)
            print("news_result", news_result)
            message_list.append(news_result)


# print("response", response)
# print(response.content)

for msg in message_list:
    msg.pretty_print()

## 4、多工具调用

In [50]:
from langchain.tools import tool
from langchain.messages import HumanMessage, AIMessage, ToolMessage
from langchain_community.chat_models import ChatZhipuAI
from dotenv import load_dotenv
import os


# 从.env文件中加载环境变量
load_dotenv(override=True)

ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")

model = ChatZhipuAI(
    model="glm-5.3-flash",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_BASE_URL
)


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取当日天气

    Args:
        city: 城市名称
    """
    return f'{city}天气晴朗'

@tool(parse_docstring=True)
def get_news() -> str:
    """
    获取当日新闻
    """
    return "近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。"

model_with_tools = model.bind_tools([get_weather, get_news])

messages = [
    HumanMessage("今天杭州天气如何？今天新闻是什么？使用已调用工具返回的结果回答，不要说无法获取实时信息，直接整理结果给我。")
]

messages.append(response)
for tool_call in response.tool_calls:
    if tool_call["name"] == "get_weather":
        tool_msg = get_weather.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    elif tool_call["name"] == "get_news":
        tool_msg = get_news.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    else:
        raise Exception("不存在的工具")

final_response = model.invoke(messages)
messages.append(final_response)

for msg in messages:
    msg.pretty_print()


python-dotenv could not parse statement starting at line 17


content='杭州天气晴朗' name='get_weather' tool_call_id='call_-7163180609848736394'
content='近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。' name='get_news' tool_call_id='call_-7163180609848736393'
================================ Human Message =================================

今天杭州天气如何？今天新闻是什么？使用已调用工具返回的结果回答，不要说无法获取实时信息，直接整理结果给我。
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_-7163180609848736394)
 Call ID: call_-7163180609848736394
  Args:
    city: 杭州
  get_news (call_-7163180609848736393)
 Call ID: call_-7163180609848736393
  Args:
================================= Tool Message =================================
Name: get_weather

杭州天气晴朗
================================= Tool Message =================================
Name: get_news

近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。
================================== Ai Message ==================================

**今日杭州天气**
- 杭州天气晴朗，适合外出活动。

**今日新闻